In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from scipy.interpolate import PchipInterpolator
from scipy.io import loadmat

from dataclasses import dataclass, replace

# set plot showing style
%config InlineBackend.figure_format = 'retina'

# load self-defined modules
%load_ext autoreload
%autoreload 2
from src.model import ModelParameters, ConsolidationParameters, HebbianSequenceModel
from src.experiment import build_exp_graph, make_trials, random_walk
from src.plotNet import make_kk_layout, plot_original_network, plot_learned_network
from src.plotBehav import plot_day1_byType, plot_day2_byType, plot_delta_byType

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
palette_whole = sns.color_palette('Paired', 12)
my_color = [palette_whole[i] for i in [0, 2, 6, 4]]
my_color_s2 = [palette_whole[i] for i in [1, 3, 7, 5]]

In [136]:
def load_design(subj):
    inFolder = 'C:\\Users\\Zhou Xiaoyue\\OneDrive - University College London\\Guenevere\\UCL\\Ph.D.project\\code.exp\\design\\'
    filename = inFolder + 'des_{}_subj.mat'.format(subj)
    des = loadmat(filename)
    
    expo_seq = des['dm_expo']['stim_seq'][0,0]
    trials_s1 = des['dm_2AFC']['seq_mat'][0,0]
    trials_s2 = des['dm_2AFC_conso']['seq_mat'][0,0]
    
    return expo_seq-1, trials_s1-1, trials_s2-1  # adjust index difference

In [208]:
def load_result(subj):
    inFolder = 'C:\\Users\\Zhou Xiaoyue\\OneDrive - University College London\\Guenevere\\UCL\\Ph.D.project\\code.exp\\data_batch2\\'

    # session 1
    filename = inFolder + 'ForcedChoice_{}_subj.mat'.format(subj)
    dat = loadmat(filename)

    rt = dat['result_2AFC']['rt'][0,0]
    answ = dat['result_2AFC']['answ'][0,0]
    resp = dat['result_2AFC']['resp'][0,0]

    result_s1 = pd.DataFrame({
        'rt': rt.ravel().astype(float),
        'answ': -1 * answ.ravel().astype(int) + 2,
        'resp': -1 * resp.ravel().astype(int) + 2
    })

    # session 2
    filename = inFolder + 'ForcedChoice_{}_subj_conso.mat'.format(subj)
    dat = loadmat(filename)

    rt = dat['result_2AFC']['rt'][0,0]
    answ = dat['result_2AFC']['answ'][0,0]
    resp = dat['result_2AFC']['resp'][0,0]

    result_s2 = pd.DataFrame({
        'rt': rt.ravel().astype(float),
        'answ': -1 * answ.ravel().astype(int) + 2,
        'resp': -1 * resp.ravel().astype(int) + 2
    })

    result_s1['session'] = 'S1'
    result_s1['trial'] = np.arange(1, 401).astype(int)
    result_s2['session'] = 'S2'
    result_s2['trial'] = np.arange(1, 401).astype(int)

    return pd.concat([result_s1, result_s2])

In [203]:
def seqMat_to_trials(seq_mat, adjacency, community):
    n_trial = seq_mat.shape[0]
    trial = np.arange(0,400)
    block = trial // 40
    cue = seq_mat[:,-2].astype(int)
    target = seq_mat[:,-1].astype(int)

    legal = adjacency[cue, target].astype(bool)
    within = community[cue] == community[target]
    sequence = seq_mat.astype(int)
    sequence = [row.tolist() for row in sequence]

    membership_lab = np.where(within, 'within', 'between')
    legal_lab = np.where(legal, 'legal', 'illegal')
    trial_type = np.char.add(membership_lab, '_')
    trial_type = np.char.add(trial_type, legal_lab)

    df = pd.DataFrame({
        'trial': trial,
        'trial_in_block': np.tile(np.arange(80), 5),
        'block': block,
        'cue': cue,
        'target': target,
        'legal': legal,
        'within': within,
        'sequence': sequence,
        'trial_type': trial_type
    })
    return df

### Simulation

In [204]:
adjacency, community, node_degree = build_exp_graph()

In [ ]:
time_ms = 6 * 60 * 1000
iti_ms = 80
tone_ms = 100
exposure_length = time_ms // (iti_ms + tone_ms)
betas = np.logspace(-2.5, 1.5, 100)


# read in subject data
subj_full = list(range(121, 155))
idx_remov = [128, 134, 140, 144, 152]
subj_list = [x for x in subj_full if x not in idx_remov]

# CHANGE HERE ---
subj = 121
expo_seq, trials_s1, trials_s2 = load_design(subj)
result = load_result(subj)

# set model parameters
beta = 0.06
params = ModelParameters(
    beta = beta,
    lr = 1.0,
    lr_test_prefix=0.0,
    lr_test_probe=0.0,
    reset_current_activity=True,
    criterion_offset=0.0
)
model = HebbianSequenceModel(
    n_nodes=adjacency.shape[0],
    params=params
)

# learn exposure
model.learn_exposure(expo_seq[0,:])
model.learn_exposure(expo_seq[1,:])

trials_s1 = seqMat_to_trials(trials_s1, adjacency, community)
trials_s2 = seqMat_to_trials(trials_s2, adjacency, community)

sim_s1 = model.run_test_phase(trials_s1)
sim_s2 = model.run_test_phase(trials_s2)


,rt,answ,resp,session,trial
0,1.184496,1,1,S1,1
1,-1.000000,1,3,S1,2
2,0.812657,0,1,S1,3
3,1.034400,0,0,S1,4
4,1.054767,0,1,S1,5
...,...,...,...,...,...
395,0.454379,0,1,S2,396
396,0.888035,0,1,S2,397
397,0.369338,0,0,S2,398
398,0.503286,0,0,S2,399


In [ ]:
# for subj in subj_list:
#     expo_seq, trials_s1, trials_s2 = load_design(subj)
#     result_s1, result_s2 = load_result(subj)

#     model = HebbianSequenceModel()
#     for ibeta, beta in enumerate(betas):
#         tbl_sim_s1 = model.run_test_phase(trails_s1)
#         tbl_sim_s2 = model.run_test_phase(trails_s2)

In [ ]:
# trial_order = ['within_legal',
#             'between_legal',
#             'within_illegal',
#             'between_illegal']
# measure = 'p_yes'
# dat4plot = (
#     sim_s1.groupby(['trial_type'], as_index=False)
#     .agg(mean_dat = (measure, 'mean'),
#          sd_dat = (measure, 'std'),
#          n_trial = (measure, 'count'))
# )

# dat4plot['trial_type'] = pd.Categorical(
#     dat4plot['trial_type'],
#     categories=trial_order,
#     ordered=True
# )

# dat4plot = dat4plot.sort_values('trial_type').reset_index(drop=True)

# # plot figures
# fig, ax = plt.subplots(figsize=(3, 2.5))
# x=np.arange(len(dat4plot))
# ax.bar(
#     x, 
#     dat4plot['mean_dat'],
#     # yerr=dat4plot['se_dat'],
#     width=0.4,
#     color=my_color,
#     capsize=5,
#     error_kw={
#         "elinewidth": 1.5,
#         "capthick": 1.5
#     },
# )
# ax.set_xticks(x)
# ax.set_xticklabels(
#     [
#         "Within-legal",
#         "Between-legal",
#         "Within-illegal",
#         "Between-illegal",
#     ],
#     rotation=20,
#     ha='right'
# )

# ax.set_xlabel("Trial type")
# ax.set_ylabel(measure)

# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)

# plt.tight_layout()